# CCDP — Car Crash Fix-Amount Predictor (capstone submission)

> **Photo → identify car → detect damage → estimate cost (USD/INR).**

This is the single notebook that drives the project. It walks through the four
variants we built (A → B → C → D), the training pipeline for each model, and
ends with a live inference demo using the released v0.2.0 weights.

## End-to-end pipeline at a glance

```mermaid
flowchart LR
    IMG[input photo] --> GATE{multi-car?}
    GATE -->|Mask R-CNN<br/>crops per car| ID[ResNet-50<br/>identifier]
    GATE -->|single car| ID
    ID -->|make / model / year| SEG_D[YOLOv8-seg<br/>damage masks]
    ID --> SEG_P[YOLOv8-seg<br/>parts masks]
    SEG_D --> INT[overlap<br/>damage ∩ parts]
    SEG_P --> INT
    INT --> SEV[severity bucket<br/>by area ratio]
    SEV --> CAT[3-tier catalog<br/>part × severity]
    CAT --> OUT[line-itemed cost<br/>USD + INR]
```

## How to read this notebook
- **§1** Setup, install, environment.
- **§1.4** Datasets used + citations.
- **§2** Identifier training (Stanford → VMMRdb). ✅ trained, 1163 classes.
- **§3** Damage segmentation (CarDD, nc=6).
- **§3b** Path A extension (CarDD + HITL). Optional; remove if not run.
- **§4** Parts segmentation (carparts, nc=15).
- **§5–§8** Variant A → B → C → D walk-through. Each shows what was added and
  what the previous variant was missing.
- **§9** Multi-car extension (Mask R-CNN gate).
- **§10** Live inference: upload image → costed line items + overlay.
- **§11** Reproducibility + final metrics table.

## Conventions
- Training cells are guarded by `RUN_TRAINING = False` with smoke defaults
  (`epochs=1, batch=2, imgsz=320`). Production values used for the released
  v0.2.0 weights are documented one line away in `# production: …` comments.
- All inference uses the released v0.2.0 weights fetched in §1.3 — no
  retraining required to run the demo.


## 1.1 Install + repo clone

In [ ]:
# If you're running this in Colab or Kaggle, clone the repo first.
# Locally, skip this cell and `cd` into the project root before launching jupyter.
import os, sys, pathlib

REPO = 'car-crash-fix-amount-predictor'
ON_COLAB  = 'google.colab' in sys.modules
ON_KAGGLE = os.path.exists('/kaggle')

if ON_COLAB or ON_KAGGLE:
    WORK = '/content' if ON_COLAB else '/kaggle/working'
    os.chdir(WORK)
    if not pathlib.Path(REPO).exists():
        os.system(f'git clone --depth 1 https://github.com/theDocWho/{REPO}.git')
    os.chdir(f'{WORK}/{REPO}')
    os.system('pip -q install -e .[ml]')

import ccdp
print('ccdp ok at', os.getcwd())

## 1.2 Environment + device check

In [ ]:
import torch
device = ('cuda' if torch.cuda.is_available()
          else 'mps' if torch.backends.mps.is_available() else 'cpu')
print('torch:', torch.__version__, 'device:', device,
      torch.cuda.get_device_name(0) if device == 'cuda' else '')

## 1.3 Fetch released production weights (v0.2.0)
All four production models are attached to the v0.2.0 GitHub release. The
inference cells in §5–§10 read from `checkpoints/production/`.

In [ ]:
import os, urllib.request, pathlib
REL = 'https://github.com/theDocWho/car-crash-fix-amount-predictor/releases/download/v0.2.0'
PROD = pathlib.Path('checkpoints/production'); PROD.mkdir(parents=True, exist_ok=True)
ASSETS = {
    'identifier.pt':   f'{REL}/identifier.pt',
    'damage_cls.pt':   f'{REL}/damage_cls.pt',
    'damage_det.pt':   f'{REL}/damage_det.pt',
    'damage_seg.pt':   f'{REL}/damage_seg.pt',
    'parts_seg.pt':    f'{REL}/parts_seg.pt',
}
for name, url in ASSETS.items():
    dst = PROD / name
    if dst.exists():
        print(f'  {name:18s} ({dst.stat().st_size/1e6:.1f} MB) — present')
        continue
    print(f'  {name:18s} downloading…')
    try:
        urllib.request.urlretrieve(url, dst)
        print(f'    -> {dst.stat().st_size/1e6:.1f} MB')
    except Exception as e:
        print(f'    SKIPPED ({e}). Inference cells will warn.')
os.system('ccdp costing init || true')

## 1.4 Datasets used

This project uses five permissively-licensed datasets. The diagram below maps
each dataset to the model(s) it trains.

```mermaid
flowchart LR
    SC[Stanford-Cars<br/>196 classes<br/>16k images]:::ds -->|warm-start| ID
    VMM[VMMRdb<br/>~9170 classes<br/>~291k images]:::ds -->|continue-train| ID[identifier.pt<br/>ResNet-50]
    CARDD[CarDD<br/>6 damage classes<br/>~4k images]:::ds --> CLS[damage_cls.pt<br/>Variant A]
    CARDD --> DET[damage_det.pt<br/>Variant B]
    CARDD --> SEG[damage_seg.pt<br/>Variants C, D]
    HITL[HITL CC0<br/>8 damage classes<br/>~1.8k images]:::ds -.-> SEG
    CP[carparts CC0<br/>15 parts<br/>~7k images]:::ds --> PARTS[parts_seg.pt<br/>Variant D]
    IAAI[IAAI auction sample<br/>year/make/model/<br/>damage location]:::ds -.->|catalog priors| CAT[3-tier<br/>cost catalog]
    classDef ds fill:#e8f4f8,stroke:#3186b3,color:#000
```

The dotted arrows (HITL → damage_seg, IAAI → catalog) indicate auxiliary uses.

### Citations

#### 1. CarDD — Car Damage Detection *(primary damage labels)*
- Kaggle mirror: [nasimetemadi/car-damage-detection](https://www.kaggle.com/datasets/nasimetemadi/car-damage-detection) · 5.7 GB · ~4,000 images
- 6 classes: `dent · scratch · crack · glass-shatter · lamp-broken · tire-flat`
- License: "other" on Kaggle; CarDD release is for academic research.

> Wang, X., Li, W., & Pan, Z. (2023). *CarDD: A New Dataset for Vision-Based Car
> Damage Detection.* IEEE Transactions on Intelligent Transportation Systems,
> 24(7), 7202–7214. <https://doi.org/10.1109/TITS.2023.3258480>

#### 2. Stanford-Cars *(make/model/year baseline)*
- Kaggle mirror: [eduardo4jesus/stanford-cars-dataset](https://www.kaggle.com/datasets/eduardo4jesus/stanford-cars-dataset) · ~2 GB · 16,185 images · 196 classes

> Krause, J., Stark, M., Deng, J., & Fei-Fei, L. (2013). *3D Object
> Representations for Fine-Grained Categorization.* 4th IEEE Workshop on 3D
> Representation and Recognition (3dRR-13). Sydney.
> <https://ai.stanford.edu/~jkrause/cars/car_dataset.html>

#### 3. VMMRdb *(make/model/year extension, ~9170 classes)*
- Kaggle CC0 mirror: [prabashwara/vmmrdb-dataset](https://www.kaggle.com/datasets/prabashwara/vmmrdb-dataset) · ~70 GB · ~291k images
- License: CC0 1.0 Universal (public domain) on the Kaggle mirror.

> Tafazzoli, F., Frigui, H., & Nishiyama, K. (2017). *A Large and Diverse
> Dataset for Improved Vehicle Make and Model Recognition.* CVPR Workshop on
> Computational Cameras and Displays. <https://github.com/faezetta/VMMRdb>

#### 4. carparts (15-class parts segmentation) *(Variant D part attribution)*
- Roboflow Universe: a CC0 polygon-annotated car-parts dataset (~7k images,
  15 canonical parts). Variant naming used: `front-bumper, rear-bumper,
  front-door, rear-door, fender, hood, trunk, roof, windshield, rear-window,
  side-mirror, headlight, taillight, wheel, grille`.
- License: CC0 (verify per snapshot on Roboflow before redistribution).

> Roboflow Universe contributors. *Car Parts Segmentation Dataset.* (CC0).
> <https://universe.roboflow.com> (search "car parts segmentation").

#### 5. HITL Vehicle Damage *(Path A extension only — §3b)*
- Kaggle CC0 mirror: human-in-the-loop curated vehicle damage segmentation
  (~1,812 images, 8 damage classes). License: CC0.

> Replace with the exact Kaggle slug once the Path A run is committed.

#### 6. IAAI Insurance Auto Auction Dataset (Rebrowser preview sample)
- Kaggle: [rebrowser/iaai-dataset](https://www.kaggle.com/datasets/rebrowser/iaai-dataset) · 11 MB · 12,353 rows (5.3% of full corpus)
- Used as priors for the parts-cost catalog (year/make/model distributions,
  primary-damage vocabulary). Premium-masked fields (`estimatedRepairCost` etc.)
  are not used for supervision.

> Rebrowser (2026). *IAAI Insurance Auto Auction Dataset (preview sample).*
> [Data set]. Kaggle. <https://www.kaggle.com/datasets/rebrowser/iaai-dataset>
> Insurance Auto Auctions (IAA), Inc. is the original source.

#### Software / framework citations
- PyTorch — Paszke et al., NeurIPS 2019.
- Ultralytics YOLOv8 (AGPL-3.0) — Jocher, Chaurasia, Qiu, 2023.
- XGBoost — Chen & Guestrin, KDD '16.
- ResNet-50 ImageNet weights — Deng et al., CVPR '09; He et al., CVPR '16.

See `CITATIONS.md` at the repo root for the canonical version of this list.


## 1.5 Dataset preview — what does the training data actually look like?

A grid of sample images per dataset, drawn live from the locally-mounted copy.
If a dataset isn't mounted locally (running the notebook in Colab without
re-downloading), the corresponding cell prints a skip message instead of failing.

This is meant to give the reviewer a visceral sense of what the models see — class
balance, image quality, framing, typical damage scale.

In [ ]:
# Small helper: show a grid of N images from a directory.
import os, glob, random
from pathlib import Path
import matplotlib.pyplot as plt
from PIL import Image

def show_grid(image_paths, title, ncols=4, figsize=(14, 7)):
    image_paths = [p for p in image_paths if Path(p).exists()][:ncols * 2]
    if not image_paths:
        print(f'[{title}] no images found — skip')
        return
    nrows = (len(image_paths) + ncols - 1) // ncols
    fig, axes = plt.subplots(nrows, ncols, figsize=figsize)
    axes = axes.flatten() if nrows * ncols > 1 else [axes]
    for ax, p in zip(axes, image_paths):
        try:
            im = Image.open(p).convert('RGB')
            ax.imshow(im); ax.set_title(Path(p).name, fontsize=8); ax.axis('off')
        except Exception as e:
            ax.set_title(f'err: {e}', fontsize=7); ax.axis('off')
    for ax in axes[len(image_paths):]:
        ax.axis('off')
    fig.suptitle(title, fontsize=12)
    plt.tight_layout(); plt.show()

random.seed(42)

### CarDD — damage segmentation source

In [ ]:
cardd_val = Path('data/raw/car-damage-detection/CarDD_release/CarDD_COCO/val2017')
if cardd_val.exists():
    samples = sorted(cardd_val.glob('*.jpg'))
    random.shuffle(samples)
    print(f'CarDD val split: {len(list(cardd_val.glob("*.jpg")))} images')
    show_grid(samples[:8], 'CarDD val samples (random 8)')
else:
    print('CarDD not mounted locally — skip preview.')
    print('(In Colab/Kaggle, the trainer will fetch it from the Kaggle mirror.)')

### Stanford-Cars — identifier baseline (196 classes)

In [ ]:
sc_train = Path('data/raw/stanford-cars-dataset/cars_train/cars_train')
if not sc_train.exists():
    sc_train = Path('data/raw/stanford-cars-dataset/cars_train')
if sc_train.exists():
    samples = sorted(sc_train.glob('*.jpg'))
    random.shuffle(samples)
    print(f'Stanford-Cars train: {len(list(sc_train.glob("*.jpg")))} images, 196 classes')
    show_grid(samples[:8], 'Stanford-Cars random samples')
else:
    print('Stanford-Cars not mounted locally — skip preview.')

### Dataset stats summary

| Dataset | Images | Classes | Used by | License |
|---|---|---|---|---|
| Stanford-Cars | 16,185 | 196 make/model/year | identifier (baseline) | research |
| VMMRdb (Kaggle CC0 mirror) | ~291,000 | **1,163** (image-bearing class folders in the Kaggle mirror) | identifier (continue-train) | CC0 |
| CarDD | ~4,000 | 6 damage | Variants A/B/C/D | research |
| carparts | ~7,000 | 15 parts | Variant D | CC0 |
| HITL (optional) | ~1,812 | 8 damage | §3b Path A | CC0 |

> **Note on VMMRdb class count.** The original VMMRdb release lists ~9,170
> classes, but the public Kaggle CC0 mirror (`prabashwara/vmmrdb-dataset`)
> contains 1,163 image-bearing class folders after the loader's
> "≥1 image per folder" filter. The released `identifier.pt` is trained on
> this 1,163-class space.


## 2. Identifier training (Stanford → VMMRdb)

**Goal:** identify the make/model/year of the photographed car so we pick the
right cost segment (a Maruti bumper ≠ a BMW bumper).

```mermaid
flowchart LR
    IM[ImageNet pretrain<br/>ResNet-50 backbone] -->|transfer| STG1[Stanford-Cars<br/>fine-tune<br/>196 classes]
    STG1 -->|warm-start<br/>swap head| STG2A[Stage 1<br/>frozen backbone<br/>2 epochs, lr 5e-4]
    STG2A --> STG2B[Stage 2<br/>unfreeze layer3/4<br/>10 epochs, lr 5e-5]
    STG2B --> OUT[identifier.pt<br/>1163 classes<br/>self-describing]
    VMM[VMMRdb Kaggle mirror<br/>1163 classes] -->|labels| STG2A
    VMM --> STG2B
```

### Head swap detail
- **Transferred** ImageNet → Stanford → VMMRdb: full backbone + `Linear(2048→512)` embedding.
- **Re-initialised** at each stage boundary: only the final `Linear(512→N)` head.
- **Catastrophic forgetting check.** After VMMRdb training we evaluate the
  *make-level* anchor accuracy on Stanford val — the predicted class's make
  token vs Stanford ground-truth make. The new label space (VMMRdb's
  `acura cl 1997` style) shares makes with Stanford's, so a working backbone
  should still nail the make even if the model/year token is wrong.

### ✅ Trained — released as v0.2.0 `identifier.pt`
Continue-train completed on Colab (T4) via
`notebooks/colab_train_identifier.ipynb`. The Kaggle CC0 mirror is smaller
than the original VMMRdb (1,163 image-bearing class folders vs the
paper's 9,170), so the released checkpoint has a 1,163-class head.


In [ ]:
# === Section 2: identifier two-stage continue-train ===
RUN_TRAINING = False

# Production values used for the released v0.2.0 weights:
#   --dataset vmmrdb --top-n 0 --batch-size 64 --num-workers 4
#   --epochs-stage1 2 --epochs-stage2 10
EPOCHS_STAGE1 = 1   # production: 2
EPOCHS_STAGE2 = 1   # production: 10
BATCH_SIZE    = 2   # production: 64
TOP_N         = 50  # production: 0  (= all ~9,170 classes)

if RUN_TRAINING:
    cmd = (f'ccdp train identifier-continue --dataset vmmrdb '
           f'--top-n {TOP_N} --batch-size {BATCH_SIZE} --num-workers 2 '
           f'--epochs-stage1 {EPOCHS_STAGE1} --epochs-stage2 {EPOCHS_STAGE2}')
    print('$', cmd); os.system(cmd)
else:
    print('Skipped — production run ~6–10h on T4. See Kaggle/Colab notebooks.')

### 2.1 What a healthy training session looks like

Per-epoch log from the actual Colab run that produced v0.2.0 `identifier.pt`:

```
[epoch 1/12]  stage=1  lr=5.00e-04
   train: loss 6.41  acc 0.014   val: loss 5.83  acc 0.046
   saved checkpoints/identifier/run_…_identifier_vmmrdb/epoch_001.pt (last, best)
[epoch 2/12]  stage=1  lr=5.00e-04
   train: loss 5.42  acc 0.082   val: loss 4.97  acc 0.143
   saved …/epoch_002.pt (last, best)
[stage 2] unfreeze layer3/layer4, trainable 23,964,711
[epoch 3/12]  stage=2  lr=5.00e-05
   train: loss 4.06  acc 0.218   val: loss 4.21  acc 0.231
…
[epoch 12/12] stage=2  lr=5.00e-05
   train: loss 1.97  acc 0.612   val: loss 2.84  acc 0.330
[anchor] Stanford make-level accuracy: 0.163
[done] best val acc: 0.3304 -> checkpoints/identifier/run_…/best.pt
```

The release flow:
```bash
# After download from Colab to ./checkpoints/identifier/identifier.pt
gh release upload v0.2.0 identifier.pt --clobber
# → Successfully uploaded 1 asset to v0.2.0
```

### 2.2 Live introspection of the released `identifier.pt`

Reads the checkpoint we just fetched in §1.3 and prints what it claims about
itself — the model card is *in* the .pt file (the `class_names`, `num_classes`,
`config`, `best_val` are all embedded).


In [ ]:
# Read the released checkpoint and surface what's baked into it.
import torch, hashlib, os
from pathlib import Path

ck_path = Path('checkpoints/production/identifier.pt')
if ck_path.exists():
    ck = torch.load(ck_path, map_location='cpu', weights_only=False)
    print(f'file: {ck_path}')
    print(f'size: {os.path.getsize(ck_path)/1e6:.1f} MB')
    print(f'sha256: {hashlib.sha256(ck_path.read_bytes()).hexdigest()}')
    print()
    print(f'num_classes: {ck["num_classes"]}')
    print(f'best val acc: {ck["best_val"]:.4f}')
    print(f'final epoch:  {ck["epoch"]}   (stage {ck["stage"]})')
    print(f'config: stage1 epochs={ck["config"]["epochs_stage1"]}, '
          f'stage2 epochs={ck["config"]["epochs_stage2"]}, '
          f'batch={ck["config"]["batch_size"]}, '
          f'lr1={ck["config"]["lr_stage1"]}, lr2={ck["config"]["lr_stage2"]}')
    print(f'\\nfirst 5 classes: {ck["class_names"][:5]}')
    print(f'last  5 classes: {ck["class_names"][-5:]}')
else:
    print('Run §1.3 first to fetch identifier.pt from the v0.2.0 release.')

### 2.3 Final metrics (release v0.2.0)

| Run | Classes | Train epochs | Val acc | Stanford make-anchor | Notes |
|---|---|---|---|---|---|
| Baseline (Stanford only) | 196 | 12 (2 + 10) | **0.77** | n/a (label space matches) | warm-start source |
| **VMMRdb continue (shipped)** | **1163** | **12 (2 + 10)** | **0.3304** | **0.163** | release v0.2.0 head-swap |

**Reading the numbers.**
- **0.33 top-1 on 1,163 classes** is ~384× chance (random = 1/1163 ≈ 0.00086).
  Reasonable for 12 epochs of continue-train on fine-grained make/model/year
  where many classes are visually near-identical (e.g.
  `acura cl 1997` vs `acura cl 1998`).
- **0.163 Stanford make-anchor** is the catastrophic-forgetting reading. It's
  lower than we'd like (the original Stanford fine-tuned head scored 0.77 on
  its own makes), but the model now answers a much harder question
  (1,163-way ≫ 196-way) and the make-only token comparison is unforgiving
  when VMMRdb's make tokenisation diverges from Stanford's. Future work:
  joint multi-task head (model + make as separate logits) to recover this.
- **Production behaviour.** For the live demo the identifier feeds the
  cost-catalog *segment* (luxury / mid-market / economy), which is derived
  from make alone — so top-1 class accuracy is more pessimistic than what
  downstream costing actually needs.

**Checkpoint provenance** — embedded in the .pt file:
- `num_classes = 1163`
- `class_names = ['acura cl 1997', 'acura cl 1998', …]` *(1163 entries)*
- `best_val = 0.3304` · `epoch = 12` · `stage = 2`
- Run with `epochs_stage1=2, epochs_stage2=10, batch_size=64, lr1=5e-4, lr2=5e-5`


## 3. Damage segmentation training (CarDD, nc=6)

**Goal:** locate damaged regions and classify the *type* of each. This is the
workhorse for Variants C and D.

### Data flow
```mermaid
flowchart LR
    RAW[CarDD COCO format<br/>polygons + labels] -->|ccdp train build-yolo-dataset| YL[YOLO format<br/>class x1 y1 x2 y2 ...]
    YL -->|images + labels split<br/>80/10/10 train/val/test| TR[YOLOv8s-seg<br/>warm-start from COCO]
    TR -->|80 epochs, batch 16,<br/>imgsz 640, patience 20| WT[damage_seg.pt<br/>box mAP50 0.712<br/>mask mAP50 0.711]
```

### Why these choices
- **`yolov8s-seg.pt`** over `n`: damage features are small, the s/n accuracy gap
  is worth the ~2× FLOPs. Anything larger overfits ~4k images.
- **imgsz 640**: smaller misses thin scratches; larger blows up training time
  on a T4 without much gain.
- **patience 20**: CarDD's val loss is wobbly — early stop too eagerly and we
  lose the last 3 mAP50 points.


### 3.1 Preview the YOLO-format labels

In [ ]:
# Show the first few label lines from a random CarDD training sample.
yolo_dir = Path('data/processed/yolo')
if yolo_dir.exists():
    label_files = list(yolo_dir.glob('labels/train/*.txt'))[:3]
    for lf in label_files:
        print(f'\n=== {lf.name} ===')
        for line in lf.read_text().splitlines()[:5]:
            parts = line.split()
            cls = int(parts[0])
            n_pts = (len(parts) - 1) // 2
            print(f'  class {cls}  ({n_pts} polygon points)  {line[:80]}…')
else:
    print('Run `ccdp train build-yolo-dataset` first to materialize the YOLO format.')
    print('Or look at the on-disk schema:')
    print('  data/processed/yolo/')
    print('    data.yaml')
    print('    images/{train,val,test}/*.jpg')
    print('    labels/{train,val,test}/*.txt   # one polygon per line')

### 3.2 Train (guarded by `RUN_TRAINING`)

In [ ]:
# === Section 3: damage YOLOv8-seg on CarDD ===
RUN_TRAINING = False

# Production values (release v0.2.0):
#   model='yolov8s-seg.pt', epochs=80, batch=16, imgsz=640, patience=20
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)

if RUN_TRAINING:
    cmd = (f"ccdp train detector --model yolov8s-seg.pt --tag damage_seg "
           f"--epochs {SMOKE['epochs']} --batch {SMOKE['batch']} "
           f"--imgsz {SMOKE['imgsz']} --patience {SMOKE['patience']}")
    print('$', cmd); os.system(cmd)
else:
    print('Skipped — production took ~2h on a T4.')

### 3.3 Expected outputs — what good training looks like

Ultralytics writes a `results.png` to the run dir with loss curves. A healthy
CarDD run shows:
- box+seg train loss falling smoothly from ~6.5 to ~1.5 over 80 epochs
- val mAP50 (box) plateauing around **0.70–0.72** by epoch 50
- val mAP50 (mask) tracks ~1% behind box mAP50 (segmentation is harder)

The cell below loads a CarDD val image and runs the released `damage_seg.pt`
on it, showing the predicted masks colour-coded by class.

In [ ]:
# Run the released damage_seg on a held-out val image, show predictions.
from PIL import Image
import matplotlib.pyplot as plt, matplotlib.patches as mpatches
import numpy as np

DAMAGE_NAMES = ['dent','scratch','crack','glass-shatter','lamp-broken','tire-flat']
COLORS = plt.get_cmap('tab10').colors

cardd_val = Path('data/raw/car-damage-detection/CarDD_release/CarDD_COCO/val2017')
weights = Path('checkpoints/production/damage_seg.pt')
sample = next(cardd_val.glob('*.jpg'), None) if cardd_val.exists() else None

if sample and weights.exists():
    from ultralytics import YOLO
    model = YOLO(str(weights))
    res = model.predict(str(sample), conf=0.25, verbose=False)[0]
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(Image.open(sample)); ax[0].set_title(f'input — {sample.name}'); ax[0].axis('off')
    overlay = np.array(Image.open(sample).convert('RGB'))
    if res.masks is not None:
        for mask, cls in zip(res.masks.data.cpu().numpy(), res.boxes.cls.cpu().numpy().astype(int)):
            m = (mask > 0.5)
            color = (np.array(COLORS[cls % len(COLORS)]) * 255).astype(np.uint8)
            overlay_resized = overlay.copy()
            # broadcast color over the mask region
            from PIL import Image as PI
            mres = np.array(PI.fromarray((m*255).astype(np.uint8)).resize(
                (overlay.shape[1], overlay.shape[0])))
            mbool = mres > 127
            overlay_resized[mbool] = (overlay_resized[mbool] * 0.5 + color * 0.5).astype(np.uint8)
            overlay = overlay_resized
    ax[1].imshow(overlay); ax[1].set_title('predicted damage masks'); ax[1].axis('off')
    patches = [mpatches.Patch(color=COLORS[i % len(COLORS)], label=n)
               for i, n in enumerate(DAMAGE_NAMES)]
    ax[1].legend(handles=patches, loc='lower right', fontsize=8)
    plt.tight_layout(); plt.show()
else:
    print('Skipping inference preview — sample image or weights missing.')

## 3b. Path A extension — CarDD + HITL (nc=6, warm-start)

> **🚧 Optional. Remove this whole section if not run.**

### Why combine CarDD with HITL?
CarDD is small (~4k images). HITL is another CC0 damage dataset (~1,812 images,
8 damage classes). Path A keeps `nc=6` and warm-starts from the released
`damage_seg.pt`, adding HITL examples for the **6 overlapping classes only**.

```mermaid
flowchart LR
    HITL[HITL labels<br/>8 classes] -->|class remap| RM[filter & relabel<br/>HITL → CarDD index]
    RM -->|drop classes<br/>with no CarDD match| FILT[~1.5k HITL imgs<br/>w/ overlapping labels]
    CDD[CarDD labels<br/>6 classes] --> COMB[combined data.yaml<br/>nc=6]
    FILT --> COMB
    COMB -->|warm-start<br/>from damage_seg.pt| TR[continue-train<br/>40 epochs]
    TR --> WT[damage_seg_pathA.pt]
```

The `HITL_TO_CARDD` remap dict is the key knob; see
`notebooks/train_damage_seg_hitl_pathA.ipynb` §4 for the editable mapping.

Path B (extend head to `nc=8` to also predict HITL's two new classes — `paint_chip`,
`rust`) is **not** pursued in this submission. The head re-init costs ~2-3× more
epochs and the catalog only prices the 6 CarDD classes anyway.

### Status
- ☐ Not yet trained → **delete this section before submission.**
- ☑ Trained → fill in §3b.2 metrics below.


### 3b.1 Combined-dataset assembly (the code, annotated)

In [ ]:
# This is the same logic that lives in train_damage_seg_hitl_pathA.ipynb §6
# — replicated here so reviewers can read it without flipping notebooks.
#
# Step A: CarDD images + labels are symlinked unchanged into a fresh tree.
# Step B: each HITL label file is rewritten — the first token per line is the
#         CarDD index from HITL_TO_CARDD (or the line is dropped if mapped to None).
#         Label files that end up empty take their image down with them.
# Step C: a fresh data.yaml is written with nc=6 and CarDD's class names.
#
# Why preserve CarDD as the canonical class list?
#   - The released damage_seg.pt was trained with these exact indices, so
#     warm-start works without any head shuffle.
#   - The catalog is keyed on these names; changing them breaks pricing.

# Verbatim from the path-A notebook (illustrative — see that notebook to run):
HITL_TO_CARDD = {
    'dent':            0,
    'scratch':         1,
    'crack':           2,
    'shattered_glass': 3,
    'broken_lamp':     4,
    'flat_tire':       5,
    'paint_chip':      None,   # dropped — no CarDD equivalent
    'rust':            None,   # dropped — no CarDD equivalent
}
mapped = sum(1 for v in HITL_TO_CARDD.values() if v is not None)
print(f'HITL classes kept: {mapped}/{len(HITL_TO_CARDD)} (rest dropped)')

### 3b.2 Train (guarded) + expected metrics

In [ ]:
RUN_TRAINING = False
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)
# Production target: epochs=40, batch=16, imgsz=640, patience=15

if RUN_TRAINING:
    from ultralytics import YOLO
    data_yaml = Path('data/processed/cardd_hitl_pathA/data.yaml')
    assert data_yaml.exists(), 'Run sections 5–6 of train_damage_seg_hitl_pathA.ipynb first.'
    model = YOLO('checkpoints/production/damage_seg.pt')
    model.train(data=str(data_yaml.resolve()), **SMOKE,
                project='checkpoints/segmenter', name='pathA_submission',
                exist_ok=True, plots=False, verbose=False)
else:
    print('Skipped. If Path A is not run, delete this whole section before submission.')

### 3b.3 Expected metrics — *to fill in if run*

| Model | Box mAP50 | Mask mAP50 | Notes |
|---|---|---|---|
| Damage seg (CarDD only, baseline) | 0.712 | 0.711 | release v0.2.0 |
| Damage seg (CarDD + HITL Path A) | TBD | TBD | warm-start, nc=6 |

Target: ≥ +1.5 mask mAP50 over baseline to justify shipping Path A.


## 4. Parts segmentation training (carparts, nc=15)

**Goal:** know *which panel* each damage mask lives on. Without this we can't
say "left front bumper severe scratch" — only "scratch somewhere on the car,"
which isn't actionable for costing.

### Canonical 15-part vocabulary

```mermaid
flowchart LR
    subgraph parts[parts_seg.pt — 15 classes]
        FB[front-bumper] --- RB[rear-bumper]
        FD[front-door] --- RD[rear-door]
        FN[fender] --- HD[hood]
        TK[trunk] --- RF[roof]
        WS[windshield] --- RW[rear-window]
        SM[side-mirror] --- HL[headlight]
        TL[taillight] --- WH[wheel]
        GR[grille]
    end
```

These are the labels the cost catalog is keyed against — `(canonical_part ×
severity_band)` → price tier.

### Result on the released weights
- Box mAP50 = **0.704**, Mask mAP50 = **0.714**


In [ ]:
# === Section 4: parts YOLOv8-seg on carparts ===
RUN_TRAINING = False
SMOKE = dict(epochs=1, batch=2, imgsz=320, patience=5)
# Production: model='yolov8s-seg.pt', epochs=80, batch=16, imgsz=640, patience=20

if RUN_TRAINING:
    from ultralytics import YOLO
    data_yaml = 'data/processed/carparts/data.yaml'
    model = YOLO('yolov8s-seg.pt')
    model.train(data=data_yaml, **SMOKE,
                project='checkpoints/segmenter', name='parts_seg',
                exist_ok=True, plots=False, verbose=False)
else:
    print('Skipped — production took ~2h on a T4.')

### 4.1 Expected output — parts mask preview

In [ ]:
# Same image as §3.3, but with parts_seg instead of damage_seg.
PART_NAMES = ['front-bumper','rear-bumper','front-door','rear-door','fender',
              'hood','trunk','roof','windshield','rear-window','side-mirror',
              'headlight','taillight','wheel','grille']

weights = Path('checkpoints/production/parts_seg.pt')
if sample and weights.exists():
    from ultralytics import YOLO
    model = YOLO(str(weights))
    res = model.predict(str(sample), conf=0.25, verbose=False)[0]
    classes_seen = sorted({int(c) for c in (res.boxes.cls.cpu().numpy() if res.boxes is not None else [])})
    print(f'Parts detected in {sample.name}: ' + ', '.join(PART_NAMES[c] for c in classes_seen))
else:
    print('Skipping — parts_seg weights or sample missing.')

## 5. Variant A — ResNet-50 multilabel damage classifier

> *The naive first pass: "what kind of damage is in this image?"*

```mermaid
flowchart LR
    IMG[image] --> RES[ResNet-50<br/>ImageNet → CarDD] --> H6[6 sigmoid heads<br/>dent / scratch / crack /<br/>glass-shatter / lamp-broken / tire-flat]
    H6 --> LK[flat catalog lookup<br/>sum prices]
    LK --> OUT[single $ estimate]
```

### What's missing
- **No location.** "Scratch present" doesn't tell you which panel.
- **No size.** Light scratch on a fender ≈ ₹1.5k. Deep scratch across the
  hood ≈ ₹18k. Can't distinguish.
- **No multi-instance.** Two dents on one panel cost more than one.


In [ ]:
# Variant A inference — schematic call. Uses `damage_cls.pt` from §1.3.
try:
    from ccdp.infer.variant_a import classify
    if sample:
        out = classify(str(sample))
        print(out)
except Exception as e:
    print(f'(Variant A inference skipped: {e})')

## 6. Variant B — YOLOv8 box detector + XGBoost on bbox features

> *"Where is the damage?" — add location and rough size.*

```mermaid
flowchart LR
    IMG[image] --> DET[YOLOv8 detector<br/>CarDD nc=6]
    DET --> BBF[bbox features<br/>per-class count<br/>total area %<br/>max box area<br/>damage spread]
    BBF --> XGB[XGBoost regressor<br/>→ predicted $ cost]
```

### Result
- **Real-detector R² = 0.736** on CarDD test, RMSE ≈ \$220.

### What's still missing
- Bbox area is a noisy proxy for damage size — a scratch and a crack with the
  same box differ wildly in cost.
- Still no part attribution.
- The regressor is a black box — hard to explain to an insurer.


In [ ]:
try:
    from ccdp.infer.variant_b import estimate_cost
    if sample:
        print(estimate_cost(str(sample)))
except Exception as e:
    print(f'(Variant B inference skipped: {e})')

## 7. Variant C — YOLOv8-seg + XGBoost on mask features

> *"How big is the damage really?" — add shape/area.*

### The lesson (this section is the core methodological insight)
Two XGBoost models, same architecture, different feature source:

| Features fed to XGBoost | R² on CarDD test |
|---|---|
| **Ground-truth polygons** | **0.939** |
| **Real model predictions** | 0.709 |

The ~0.23 R² gap is the noise of the segmenter's mask quality. Masks **are**
more informative than bboxes — the GT score proves it — but the real masks
aren't clean enough to beat Variant B (0.736). The regressor's error eats
the gain.

```mermaid
flowchart TB
    GT[ground-truth<br/>CarDD polygons] --> XGB_GT[XGBoost on mask<br/>area features]
    XGB_GT --> R1[R² = 0.939 ✅]
    PRED[YOLOv8-seg<br/>predicted masks] --> XGB_PR[XGBoost on mask<br/>area features]
    XGB_PR --> R2[R² = 0.709 ❌]
    R1 -.->|gap of 0.23<br/>= segmenter noise| R2
```

**Takeaway:** *how you use the information matters as much as having it.* So
we changed *how* we use the masks → Variant D. We don't ask a regressor to
collapse them into one number; we use them directly to produce auditable
line items.


## 8. Variant D — Production pipeline (catalog line items)

> *Use the masks directly. Skip the regressor entirely.*

```mermaid
flowchart LR
    IMG[image] --> SD[damage_seg.pt<br/>nc=6]
    IMG --> SP[parts_seg.pt<br/>nc=15]
    SD --> INT[per-pixel overlap<br/>damage_mask ∩ part_mask]
    SP --> INT
    INT --> TUP["(canonical_part,<br/>damage_type, area_px)"]
    TUP --> SEV{area / car_area}
    SEV -->|≥ 0.06| SV[severe]
    SEV -->|≥ 0.015| MD[moderate]
    SEV -->|< 0.015| MN[minor]
    SV --> CAT[3-tier catalog<br/>part × severity]
    MD --> CAT
    MN --> CAT
    CAT --> OUT[line items + total $]
```

### Why this beats Variant C
- **No regressor.** No black-box error.
- **Per-line-item output.** Each row is a damaged part + price — *"front
  bumper, moderate crack, ₹14,500"* — auditable by a human adjuster.
- **Masks don't have to be perfect.** They just have to fire in the right
  region. Even a 0.71 mAP50 mask is plenty for "this damage overlaps the
  front-bumper polygon → flag that part."

### A → B → C → D recap
| Variant | What it added | What it still missed |
|---|---|---|
| A — classifier  | damage **type** | location, size, instance count |
| B — det + xgb   | location (bbox), rough size | shape, part, opaque cost |
| C — seg + xgb   | **shape** (mask) | same regressor limit; opaque |
| D — seg ∩ parts | **part attribution, severity, auditable line items** | (shipped) |


In [ ]:
try:
    from ccdp.infer.variant_d import estimate
    if sample:
        report = estimate(str(sample))
        for line in report.get('line_items', []):
            print(f"  {line['part']:18s} {line['damage_type']:14s} "
                  f"{line['severity']:8s} ${line['cost_usd']:.0f}")
        print(f"\nTotal: ${report.get('total_usd', 0):.0f}")
except Exception as e:
    print(f'(Variant D inference skipped: {e})')

## 9. Multi-car extension

A single photo might show two cars in a collision. The Variant D pipeline runs
**per car**: a Mask R-CNN gate (`detect_all`) crops each car instance, then
identification + Variant D run on each crop independently.

```mermaid
flowchart LR
    IMG[collision photo] --> MR[Mask R-CNN<br/>car instance segmentation]
    MR --> CROP1[crop: car 1]
    MR --> CROP2[crop: car 2]
    CROP1 --> P1[identify +<br/>Variant D]
    CROP2 --> P2[identify +<br/>Variant D]
    P1 --> AGG[per-vehicle aggregation]
    P2 --> AGG
    AGG --> RPT[multi-vehicle report]
```

The final report breaks down per vehicle so the adjuster sees, e.g.,
*"Vehicle 1 (Honda City): ₹38k. Vehicle 2 (Maruti Swift): ₹22k."*


In [ ]:
try:
    from ccdp.infer.multi_car import estimate_multi
    if sample:
        print(estimate_multi(str(sample)))
except Exception as e:
    print(f'(multi-car inference skipped: {e})')

## 10. Live inference demo

Upload a damage photo. The pipeline runs all stages and returns the
line-itemed cost report in both USD and INR.

> Run §1.1–§1.3 first to fetch the release weights.

In [ ]:
# Upload widget — works on Colab. Plain Jupyter: set IMG_PATH manually.
import os, sys, pathlib
IMG_PATH = None

if IMG_PATH is None and 'google.colab' in sys.modules:
    from google.colab import files
    uploaded = files.upload()
    IMG_PATH = next(iter(uploaded))
elif IMG_PATH is None:
    # Fall back to a CarDD val image if nothing uploaded.
    cardd_val = pathlib.Path('data/raw/car-damage-detection/CarDD_release/CarDD_COCO/val2017')
    if cardd_val.exists():
        IMG_PATH = str(next(cardd_val.glob('*.jpg')))
    else:
        print('Set IMG_PATH to a local image path and re-run.')
        IMG_PATH = ''

if IMG_PATH:
    print('inference image:', IMG_PATH)
    assert pathlib.Path(IMG_PATH).exists(), f'no file at {IMG_PATH}'

In [ ]:
# End-to-end Variant D + multi-car.
import json
try:
    from ccdp.infer.multi_car import estimate_multi
    report = estimate_multi(IMG_PATH)
    print(json.dumps(report, indent=2, default=str)[:2000])
except Exception as e:
    print(f'inference error: {e}')
    report = {}

In [ ]:
# Pretty-print line items + INR conversion.
USD_TO_INR = 83.0
vehicles = report.get('vehicles', [report]) if report else []
for i, veh in enumerate(vehicles, start=1):
    ident = veh.get('identification', {})
    print(f'\n=== Vehicle {i}: {ident.get("class", "unknown")} ===')
    total_usd = 0.0
    for line in veh.get('line_items', []):
        usd = float(line['cost_usd']); total_usd += usd
        inr = usd * USD_TO_INR
        print(f"  {line['part']:18s} {line['damage_type']:14s} "
              f"{line['severity']:8s}  ${usd:>7.0f}  ₹{inr:>9,.0f}")
    print(f"  {'TOTAL':36s}  ${total_usd:>7.0f}  ₹{total_usd*USD_TO_INR:>9,.0f}")

In [ ]:
# Overlay visualization: original + damage masks + parts masks.
import matplotlib.pyplot as plt
from PIL import Image
try:
    from ccdp.infer.seg_inference import visualize_overlay
    fig, ax = plt.subplots(1, 2, figsize=(14, 7))
    ax[0].imshow(Image.open(IMG_PATH)); ax[0].set_title('input'); ax[0].axis('off')
    ax[1].imshow(visualize_overlay(IMG_PATH)); ax[1].set_title('damage ∩ parts overlay'); ax[1].axis('off')
    plt.tight_layout(); plt.show()
except Exception as e:
    print(f'(overlay skipped: {e})')

## 11. Reproducibility + final metrics

### Released v0.2.0 model card

| Model | Variant role | Backbone | Train data | Val metric | Notes |
|---|---|---|---|---|---|
| `identifier.pt` | identification | ResNet-50 | Stanford-Cars → VMMRdb (Kaggle CC0 mirror) | 1163-class val acc **0.3304**, Stanford make-anchor **0.163** | self-describing (1,163 classes embedded) · 101 MB · sha256 `01393c2e…` |
| `damage_cls.pt` | Variant A | ResNet-50 | CarDD | multilabel acc ≈ 0.78 | 6 sigmoid heads |
| `damage_det.pt` | Variant B | YOLOv8s | CarDD | box mAP50 ≈ 0.70 | box detector |
| `damage_seg.pt` | Variants C, D | YOLOv8s-seg | CarDD | box mAP50 0.712 / mask 0.711 | masks |
| `parts_seg.pt`  | Variant D    | YOLOv8s-seg | carparts | box mAP50 0.704 / mask 0.714 | 15-class |

### Variant comparison (CarDD test split)

| Variant | Approach | Metric | Score |
|---|---|---|---|
| A | ResNet-50 multilabel + flat catalog | per-type acc | ~0.78 |
| B | YOLOv8 det + XGBoost on bbox feats | R² (cost) | 0.736 |
| C — GT polygons | YOLOv8-seg + XGBoost on mask feats | R² (cost) | **0.939** ¹ |
| C — real masks  | same, real model preds | R² (cost) | 0.709 |
| D — production | seg ∩ parts → catalog line items | line-item match | auditable, no R² applies |

¹ The 0.939 → 0.709 gap is the *core lesson* of this project: mask information
beats bbox information when masks are clean. Real masks aren't clean enough,
so the regressor's error eats the gain. Variant D side-steps the regressor entirely.

### How to reproduce end-to-end (production values)
```bash
# 1. Identifier two-stage (Stanford → VMMRdb continue) — 12 epochs total
ccdp train identifier-continue --dataset vmmrdb --top-n 0 \
    --batch-size 64 --num-workers 4 --epochs-stage1 2 --epochs-stage2 10
# Released v0.2.0 numbers: 1163-class val acc 0.3304, Stanford make-anchor 0.163.

# 2. Damage seg (CarDD)
ccdp train detector --model yolov8s-seg.pt --tag damage_seg \
    --epochs 80 --batch 16 --imgsz 640 --patience 20

# 3. Parts seg (carparts) — see §4
# 4. (Optional) Path A — see notebooks/train_damage_seg_hitl_pathA.ipynb
```

### Links
- **Code:** <https://github.com/theDocWho/car-crash-fix-amount-predictor>
- **Weights:** the v0.2.0 release on the same repo (includes the new VMMRdb
  `identifier.pt` uploaded via `gh release upload v0.2.0 identifier.pt --clobber`)
- **Live demo:** HuggingFace Space — see repo README
- **Canonical citations:** `CITATIONS.md` at the repo root
